# 06 – File Handling & Exception Management

Topics covered:
- Reading and writing text files
- Context managers (`with` statement)
- Working with CSV and JSON files
- Exception handling (`try` / `except` / `else` / `finally`)
- Raising and creating custom exceptions
- Best practices

## 0. From-scratch: what `with` automates

A resource that must be released after use (a file handle, a lock, a DB connection) has one
sharp edge: if the code between "acquire" and "release" raises, and "release" was only written
as the last line of a function, that last line is skipped — the resource leaks. The cells below
build this failure for real with a tiny stand-in resource, fix it with a manual `try`/`finally`,
then wrap that pattern into a hand-rolled context-manager class (`__enter__`/`__exit__`) — the
mechanism `with open(...)` runs underneath, used explicitly here *before* section 1 uses it.

In [1]:
# A tiny stand-in "resource" that must be closed after use — like a file handle.
# `open_resources` tracks everything currently open, so a leak is directly observable.
open_resources = []


class ManagedResource:
    def __init__(self, name):
        self.name = name
        open_resources.append(name)
        print(f"opened {name}")

    def do_work(self, should_fail=False):
        if should_fail:
            raise RuntimeError(f"something went wrong while using {self.name}")
        print(f"using {self.name}")

    def close(self):
        open_resources.remove(self.name)
        print(f"closed {self.name}")


# --- The bug: close() as the last line, unprotected ---
def naive_use(should_fail):
    r = ManagedResource("naive.txt")
    r.do_work(should_fail=should_fail)   # if this raises, the line below never runs
    r.close()


open_resources.clear()
try:
    naive_use(True)
except RuntimeError as e:
    print("caught:", e)
print("leaked resources after naive_use:", open_resources)

opened naive.txt
caught: something went wrong while using naive.txt
leaked resources after naive_use: ['naive.txt']


In [2]:
# --- The fix: manual try/finally — close() runs even though the exception still propagates ---
def safe_use(should_fail):
    r = ManagedResource("safe.txt")
    try:
        r.do_work(should_fail=should_fail)
    finally:
        r.close()


open_resources.clear()
try:
    safe_use(True)
except RuntimeError as e:
    print("caught:", e)
print("leaked resources after safe_use:", open_resources)

opened safe.txt
closed safe.txt
caught: something went wrong while using safe.txt
leaked resources after safe_use: []


`try`/`finally` fixes the leak, but it has to be re-written correctly at *every* call site that
uses a `ManagedResource` — easy to get right once, easy to forget the tenth time. Wrapping
"acquire, then guarantee release" into a class with `__enter__`/`__exit__` moves that guarantee
into one place, used with the `with` statement instead of hand-written `try`/`finally` everywhere.

In [3]:
# --- A hand-rolled context manager: the try/finally moves inside __enter__/__exit__ ---
class ManagedResourceCM:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.resource = ManagedResource(self.name)
        return self.resource          # this becomes the `as r` value

    def __exit__(self, exc_type, exc_value, traceback):
        self.resource.close()         # runs no matter how the `with` block exits
        return False                  # False = don't suppress the exception; let it propagate


open_resources.clear()
try:
    with ManagedResourceCM("cm.txt") as r:
        r.do_work(should_fail=True)
except RuntimeError as e:
    print("caught:", e)
print("leaked resources after the `with` block:", open_resources)

opened cm.txt
closed cm.txt
caught: something went wrong while using cm.txt
leaked resources after the `with` block: []


This is exactly what `with open(path) as f:` does underneath: `open()` returns a file object whose
`__enter__` returns itself and whose `__exit__` calls `.close()` unconditionally — including when
the block raises. Section 7 later in this notebook shows `contextlib.contextmanager`, which
generates this same `__enter__`/`__exit__` pair from a single generator function instead of a
full class. The rest of the notebook uses `with open(...)` directly, now that its mechanism is
explicit.

## 1. Reading and Writing Text Files

Always use the `with` statement — it guarantees the file is closed even if an exception occurs.

In [4]:
# Write a file
with open("/tmp/sample.txt", "w") as f:
    f.write("Line 1: Hello, Python!\n")
    f.write("Line 2: File handling is easy.\n")
    f.write("Line 3: Always use 'with'.\n")

print("File written.")

File written.


In [5]:
# Read entire file at once
with open("/tmp/sample.txt", "r") as f:
    content = f.read()
print(content)

Line 1: Hello, Python!
Line 2: File handling is easy.
Line 3: Always use 'with'.



In [6]:
# Read line by line — memory-efficient for large files
with open("/tmp/sample.txt") as f:
    for i, line in enumerate(f, 1):
        print(f"{i}: {line}", end="")

1: Line 1: Hello, Python!
2: Line 2: File handling is easy.
3: Line 3: Always use 'with'.


In [7]:
# Read into a list of lines
with open("/tmp/sample.txt") as f:
    lines = f.readlines()   # includes '\n'
print(lines)

# Strip newlines
stripped = [l.rstrip() for l in lines]
print(stripped)

['Line 1: Hello, Python!\n', 'Line 2: File handling is easy.\n', "Line 3: Always use 'with'.\n"]
['Line 1: Hello, Python!', 'Line 2: File handling is easy.', "Line 3: Always use 'with'."]


In [8]:
# File modes
# 'r'  — read (default)
# 'w'  — write (creates or overwrites)
# 'a'  — append (creates or appends)
# 'x'  — exclusive create (fails if file exists)
# 'rb' — read binary
# 'w+' — read + write

# Append to file
with open("/tmp/sample.txt", "a") as f:
    f.write("Line 4: Appended line.\n")

with open("/tmp/sample.txt") as f:
    print(f.read())

Line 1: Hello, Python!
Line 2: File handling is easy.
Line 3: Always use 'with'.
Line 4: Appended line.



## 2. Working with CSV Files

In [9]:
import csv

# Write CSV
students = [
    {"name": "Alice", "age": 22, "score": 92},
    {"name": "Bob",   "age": 24, "score": 88},
    {"name": "Carol", "age": 23, "score": 95},
]

with open("/tmp/students.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "score"])
    writer.writeheader()
    writer.writerows(students)

print("CSV written")

CSV written


In [10]:
# Read CSV
with open("/tmp/students.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(dict(row))

{'name': 'Alice', 'age': '22', 'score': '92'}
{'name': 'Bob', 'age': '24', 'score': '88'}
{'name': 'Carol', 'age': '23', 'score': '95'}


## 3. Working with JSON Files

In [11]:
import json

config = {
    "model": "random_forest",
    "n_estimators": 100,
    "max_depth": 5,
    "features": ["age", "income", "credit_score"]
}

# Save
with open("/tmp/config.json", "w") as f:
    json.dump(config, f, indent=2)

# Load
with open("/tmp/config.json") as f:
    loaded_config = json.load(f)

print(loaded_config)
print(loaded_config["features"])

{'model': 'random_forest', 'n_estimators': 100, 'max_depth': 5, 'features': ['age', 'income', 'credit_score']}
['age', 'income', 'credit_score']


## 4. Exception Handling

```
try:
    risky code
except SomeError as e:
    handle error
else:
    runs only if NO exception
finally:
    ALWAYS runs (cleanup)
```

In [12]:
# Basic exception handling
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None
    else:
        print(f"{a} / {b} = {result}")
        return result
    finally:
        print("divide() finished")  # runs regardless

divide(10, 2)
print("---")
divide(10, 0)

10 / 2 = 5.0
divide() finished
---
Cannot divide by zero!
divide() finished


In [13]:
# Catching multiple exceptions
def safe_convert(value, target_type):
    try:
        return target_type(value)
    except (ValueError, TypeError) as e:
        print(f"Conversion failed: {e}")
        return None

print(safe_convert("42", int))     # 42
print(safe_convert("hello", int))  # error
print(safe_convert(None, float))   # error

42
Conversion failed: invalid literal for int() with base 10: 'hello'
None
Conversion failed: float() argument must be a string or a real number, not 'NoneType'
None


In [14]:
# Common built-in exceptions
exceptions = [
    (ValueError,       lambda: int("abc")),
    (TypeError,        lambda: "a" + 1),
    (IndexError,       lambda: [1,2,3][10]),
    (KeyError,         lambda: {}["x"]),
    (AttributeError,   lambda: None.upper()),
    (FileNotFoundError,lambda: open("missing.txt")),
    (ZeroDivisionError,lambda: 1/0),
    (OverflowError,    lambda: 10.0**10000),
]

for exc_type, trigger in exceptions:
    try:
        trigger()
    except exc_type as e:
        print(f"{exc_type.__name__}: {e}")

ValueError: invalid literal for int() with base 10: 'abc'
TypeError: can only concatenate str (not "int") to str
IndexError: list index out of range
KeyError: 'x'
AttributeError: 'NoneType' object has no attribute 'upper'
FileNotFoundError: [Errno 2] No such file or directory: 'missing.txt'
ZeroDivisionError: division by zero
OverflowError: (34, 'Numerical result out of range')


In [15]:
# Exception hierarchy — catch specific before general
def read_file_safe(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {path}")
    except PermissionError:
        print(f"No permission to read: {path}")
    except OSError as e:
        print(f"OS error: {e}")
    return None

content = read_file_safe("/tmp/sample.txt")   # works
content = read_file_safe("/tmp/missing.txt")  # caught cleanly

File not found: /tmp/missing.txt


## 4b. Failure mode: swallowing exceptions silently

A bare `except:` (or `except Exception:` used as a reflex) catches *everything*, including bugs
that have nothing to do with the operation being guarded — a typo, a wrong type, a `None` where
a number was expected. When the handler then returns a plausible-looking default, the bug doesn't
crash; it produces a wrong answer that looks like a valid result.

In [16]:
# BAD: this was written to guard against ZeroDivisionError only, but "except:" catches everything.
def bad_average(values):
    try:
        return sum(values) / len(values)
    except:                      # bare except — catches ZeroDivisionError AND real bugs alike
        return 0


print("bad_average([]) =", bad_average([]))                 # intended case: empty list -> 0, fine
print("bad_average([10, '20', 30]) =", bad_average([10, "20", 30]))
# ^ a real bug (mixed str/int -> TypeError on sum()) is silently reported as 0,
#   indistinguishable from the legitimate "empty list" case above.

print()

# GOOD: catch only what you actually mean to handle; real bugs still surface.
def good_average(values):
    try:
        return sum(values) / len(values)
    except ZeroDivisionError:
        return 0


print("good_average([]) =", good_average([]))
try:
    good_average([10, "20", 30])
except TypeError as e:
    print("good_average([10, '20', 30]) raised (as it should):", e)

bad_average([]) = 0
bad_average([10, '20', 30]) = 0

good_average([]) = 0
good_average([10, '20', 30]) raised (as it should): unsupported operand type(s) for +: 'int' and 'str'


## 5. Raising Exceptions

In [17]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age must be 0-150, got {age}")
    return age

try:
    set_age("thirty")
except TypeError as e:
    print(e)

try:
    set_age(200)
except ValueError as e:
    print(e)

age must be int, got str
age must be 0-150, got 200


## 6. Custom Exceptions

Create domain-specific exceptions by subclassing `Exception`.

In [18]:
class ModelNotTrainedError(Exception):
    """Raised when prediction is attempted before training."""
    pass


class DataValidationError(ValueError):
    """Raised when input data fails validation."""
    def __init__(self, column, message):
        self.column = column
        super().__init__(f"Column '{column}': {message}")


class SimpleClassifier:
    def __init__(self):
        self._trained = False

    def predict(self, X):
        if not self._trained:
            raise ModelNotTrainedError("Call .fit() before .predict()")
        return [0] * len(X)


clf = SimpleClassifier()
try:
    clf.predict([1, 2, 3])
except ModelNotTrainedError as e:
    print(f"ModelNotTrainedError: {e}")

try:
    raise DataValidationError("age", "contains negative values")
except DataValidationError as e:
    print(f"DataValidationError: {e} (column={e.column})")

ModelNotTrainedError: Call .fit() before .predict()
DataValidationError: Column 'age': contains negative values (column=age)


## 7. Context Managers

The `with` statement works with any object that implements `__enter__` and `__exit__`.  
Build your own with `contextlib`.

In [19]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label=""):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label} took {elapsed:.4f}s")

with timer("list comprehension"):
    squares = [x**2 for x in range(1_000_000)]

list comprehension took 0.0409s


## Best Practices

| Practice | Reason |
|----------|--------|
| Always use `with` for files | Guarantees file is closed, even on exception |
| Catch specific exceptions | `except Exception` hides bugs |
| Don't swallow exceptions silently | At minimum, log the error |
| Use `else` for success path | Keeps try block minimal |
| Use `finally` for cleanup | Runs regardless of success/failure |
| Raise early, catch late | Validate inputs at function entry |

**Next →** [07 – OOP](../07-oops/)